In [ ]:
from pathlib import Path
from collections import defaultdict
import rasterio
from rasterio.merge import merge

base_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence")
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022,2023,2024, 2025, 2026]

In [10]:
import re

ASCENDING_ORBITS = {15, 44, 117}
DESCENDING_ORBITS = {95, 168}


def merge_swaths(files, output_filename):
    datasets = [rasterio.open(f) for f in files]
    try:
        mosaic, transform = merge(datasets, method="max", nodata=0)

        meta = datasets[0].meta.copy()
        meta.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=transform,
            count=1,
            nodata=0,
            dtype=mosaic.dtype,
        )

        with rasterio.open(output_filename, "w", **meta) as dst:
            dst.write(mosaic[0], 1)

        print(f"  Saved: {output_filename}")
    finally:
        for ds in datasets:
            ds.close()


def create_mosaics(year):
    output_folder = base_folder / str(year)
    files = list(output_folder.glob(f"sar_coherence_T*_{year}.tiff"))

    ascending_files = []
    descending_files = []

    for f in files:
        match = re.search(r"T(\d+)", f.name)
        if not match:
            continue
        orbit = int(match.group(1))

        if orbit in ASCENDING_ORBITS:
            ascending_files.append(f)
        elif orbit in DESCENDING_ORBITS:
            descending_files.append(f)

    print(f"{year} | Ascending: {len(ascending_files)} files, "
          f"Descending: {len(descending_files)} files")

    if ascending_files:
        merge_swaths(ascending_files, output_folder / f"coh_mosaic_ascending_{year}.tiff")
    else:
        print(f"  {year}: no ascending composites found, skipping")

    if descending_files:
        merge_swaths(descending_files, output_folder / f"coh_mosaic_descending_{year}.tiff")
    else:
        print(f"  {year}: no descending composites found, skipping")


for year in years:
    #process_year(year)
    create_mosaics(year)

2016 | Ascending: 14 files, Descending: 11 files
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/coh_mosaic_ascending_2016.tiff
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2016/coh_mosaic_descending_2016.tiff
2017 | Ascending: 14 files, Descending: 11 files
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/coh_mosaic_ascending_2017.tiff
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017/coh_mosaic_descending_2017.tiff
2018 | Ascending: 14 files, Descending: 11 files
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2018/coh_mosaic_ascending_2018.tiff
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2018/coh_mosaic_descending_2018.tiff
2019 | Ascending: 14 files, Descending: 11 files
  Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2019/coh_mosaic_ascending_2019.tiff
  S